# Meta-Llama-3-8B × opc-sft-stage2 leaderboard — long-horizon (r=256, 9000 steps)

Scale-up cell of the polar-product campaign. Meta-Llama-3-8B (base) × `opc-sft-stage2` (Magicoder) × r=256 × **global_batch=16 (batch=2 × accum=8 — 8B needs bs2 for 96GB-Blackwell headroom; clean-k2 peaks ~61GB at bs2 vs ~92GB at bs4)** × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell.

Data reuses the `_llama32` packed corpus (Llama-3/3.1/3.2 share an identical token id-mapping — verified). Speed metric = fraction of the 9000-step horizon a method needs to reach the best AdamW final eval_loss at this cell (lower = better). Runs in flight render partially (panels use `allow_partial=True`).

Arms: **AdamW** baseline (lr grid 1e-5..3e-4) · **chord-tight-clean k=2 gram ns=8** (lr grid 3e-3..1e-1, primary) · optional plain chord-tight k=1 secondary.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Repo root by marker-walk, so this notebook works from any subdir.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lora_playground').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.plotting import leaderboard_panel, canonical_label

# Run membership comes from the shared registry (lora_playground.workloads) — the
# SAME source the leaderboard doc uses. Cell: Meta-Llama-3-8B / opc / r256.
MODEL, DATASET, RANK = 'meta-llama/Meta-Llama-3-8B', 'opc', 256

def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def is_curv(label):
    return ('SOAP-curv' in label) or ('KL-Shampoo' in label) or ('+curv' in label)

## r=256 — all arms (AdamW baseline + chord-tight)

Every completed/in-flight variant at this cell. In-flight runs show from their first eval.

In [ ]:
_fig, _tdf, sdf = leaderboard_panel(MODEL, DATASET, RANK,
    'Meta-Llama-3-8B × opc-sft-stage2 × r=256 × 9000 steps — all arms (bs2/ga8)',
    figsize=(13, 5))
plt.show()
sdf

## r=256 — chord-tight-clean k=2 (primary) vs AdamW

In [ ]:
_fig, _tdf, sdf = leaderboard_panel(MODEL, DATASET, RANK,
    'Meta-Llama-3-8B × opc × r=256 — chord-tight-clean k=2 vs AdamW',
    label_filter=lambda l, c: l == 'AdamW' or (picard_of(c) == 2 and not is_curv(l)),
    figsize=(13, 5))
plt.show()
sdf

## r=256 — chord-tight k=1 (plain, secondary) vs AdamW

In [ ]:
_fig, _tdf, sdf = leaderboard_panel(MODEL, DATASET, RANK,
    'Meta-Llama-3-8B × opc × r=256 — chord-tight k=1 (plain) vs AdamW',
    label_filter=lambda l, c: l == 'AdamW' or (picard_of(c) == 1 and not is_curv(l)),
    figsize=(13, 5))
plt.show()
sdf